<a href="https://colab.research.google.com/github/ZainabFatima-hzf/ML-flyRank/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZainabFatima-hzf/ML-flyRank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Lane recap:** Growth / Recovery / Momentum Prediction — will a page decline, recover, or gain
momentum *next*, using only what was knowable before that point.

**Contract, plain words (answers 1-2 of 5; fields + label + exclusion follow in Section 2):**

1. **What one row means:** one row = one content item, on one calendar day —
   `content_hash_id + report_date` (this is a "content-day," the grain of
   `fact_content_daily_performance`). I am NOT flattening this to one-row-per-page yet; I stay at
   the daily grain through Section 3 so I can build prior-window features honestly (see Section 4
   trap), then I'll aggregate to one-row-per-decision only when I build the actual feature frame.
2. **Which table(s):** `fact_content_daily_performance` (the time series) joined to `dim_content`
   (content-level metadata, joined on `content_hash_id`) and `dim_clients` (per-client history
   coverage — `gsc_data_start`, `ga4_data_start` — joined on `client_hash_id`). I am not touching
   `fact_content_query_90d` this week; its 90-day window overlaps the daily table's final months,
   which is exactly the kind of window-alignment problem I want to avoid until I need query-mix
   features specifically.
3. **Which time window:** I develop and verify everything on the **mid-panel month `2026-03`**
   (`report_date` between `2026-03-01` and `2026-03-31`). I am deliberately NOT touching
   `fact_content_daily_performance_sample` (June 2026) for anything label-related — that table
   *is* the final month of the whole panel, i.e. the natural "future" of any past→future label. If
   I develop logic there, I am peeking at my own test window.

Verification for point 1 (grain) and the row count / date span for point 3 are Queries 1 and 2 in
Section 3 — I don't just assert the grain here, I check it there.

In [1]:
# --- Setup: DuckDB reads the Hugging Face parquet release directly, no full download needed ---
# Run this cell first. It needs HF_TOKEN in Colab Secrets (key icon, left sidebar) --
# never paste the token into a cell; this repo is public.

!pip install -q duckdb huggingface_hub

import duckdb, os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.sql(f"""
    CREATE OR REPLACE SECRET hf_token (
        TYPE huggingface,
        TOKEN '{os.environ["HF_TOKEN"]}'
    )
""")

REPO         = "hf://datasets/FlyRank/internship-warehouse"
DIM_CONTENT  = f"{REPO}/dim_content.parquet"
DIM_CLIENTS  = f"{REPO}/dim_clients.parquet"
QUERY_90D    = f"{REPO}/fact_content_query_90d.parquet"
DAILY        = f"{REPO}/fact_content_daily_performance/*/*.parquet"
DAILY_SAMPLE = f"{REPO}/fact_content_daily_performance_sample.parquet"  # sealed June test month -- never develop label logic on this

# --- Search before assuming: list what's actually in the release before hardcoding column names ---
from huggingface_hub import HfApi
api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
print(f"{len(files)} files in the release:")
for f in sorted(files):
    print(" ", f)


24 files in the release:
  .gitattributes
  README.md
  dim_clients.parquet
  dim_content.parquet
  fact_content_daily_performance/month=2025-01/data_0.parquet
  fact_content_daily_performance/month=2025-02/data_0.parquet
  fact_content_daily_performance/month=2025-03/data_0.parquet
  fact_content_daily_performance/month=2025-04/data_0.parquet
  fact_content_daily_performance/month=2025-05/data_0.parquet
  fact_content_daily_performance/month=2025-06/data_0.parquet
  fact_content_daily_performance/month=2025-07/data_0.parquet
  fact_content_daily_performance/month=2025-08/data_0.parquet
  fact_content_daily_performance/month=2025-09/data_0.parquet
  fact_content_daily_performance/month=2025-10/data_0.parquet
  fact_content_daily_performance/month=2025-11/data_0.parquet
  fact_content_daily_performance/month=2025-12/data_0.parquet
  fact_content_daily_performance/month=2026-01/data_0.parquet
  fact_content_daily_performance/month=2026-02/data_0.parquet
  fact_content_daily_performance/m

## 2. Fields: feature / label / context / excluded

**Contract answers 4-5 (label/proxy, and the one thing I deliberately exclude):**

- **Context (join/group/split only, never features):** `content_hash_id`, `client_hash_id`,
  `url_hash_id`, `keyword_hash_id`, `report_date` itself. These identify *which* row this is, not
  a signal about the page.
- **Feature (knowable at the decision moment — "prior window" signals):** GSC signals
  (`gsc_impressions`, `gsc_clicks`, `gsc_avg_position`) and GA4 signals (`ga4_sessions`, engaged
  sessions, `scroll_events`) **aggregated only over days strictly before the decision date**;
  content-level fields from `dim_content` that don't change after publication (e.g.
  `content_created_date` → content age at decision time); `ga4_data_available` (a coverage flag,
  safe to use as-is since it's known immediately).
- **Label / proxy (the thing I predict — never a feature):** a forward-looking decline/
  recovery/momentum flag built by comparing a **prior window** (e.g. 28 days before the decision
  date) to a **later window** (e.g. 28 days after it), both taken from
  `fact_content_daily_performance` at the daily grain. This is NOT built yet in Week 3 — Section 3
  only proves the daily table can support it. I am explicitly not reusing the starter CSV's
  `is_declining_label`/`trend_direction` here, because on warehouse data I have real daily rows
  and can build an actual forward window instead of another current-window proxy.
- **Excluded, on purpose:** FlyRank's own product decision fields (`health_score`,
  `priority_score`, `action_type`, refresh flags, `is_quick_win`) are excluded — the
  `flyrank-context` skill and lane guide are explicit that these are the product's own answers, not
  observable evidence, and using them would make any result circular (the model would just be
  learning to copy a decision someone already made, not finding real signal).

In [2]:
# Discovery: real columns, confirmed against the actual files (not guessed).
# fact_content_daily_performance: report_date, client_hash_id, content_hash_id, client_has_gsc,
#   client_has_ga4, gsc_data_available, ga4_data_available, gsc_impressions, gsc_clicks,
#   gsc_sum_position, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_users,
#   ga4_engaged_sessions, ga4_total_engagement_sec, sessions_organic/direct/referral/social/paid/ai,
#   ai_chatgpt/perplexity/gemini/copilot/claude/meta/other, scroll_events, month
# dim_content: client_hash_id, content_hash_id, keyword_hash_id, url_hash_id, ...,
#   content_created_date, content_updated_date, content_type, is_published, is_deleted, ...
# dim_clients: client_hash_id, is_active, has_gsc_access, has_ga4_access, access_profile,
#   client_created_date, client_updated_date, gsc_data_start, ga4_data_start

schema_daily = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{DAILY}') LIMIT 0").df()
print("fact_content_daily_performance columns:")
print(schema_daily["column_name"].tolist())

schema_content = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{DIM_CONTENT}') LIMIT 0").df()
print()
print("dim_content columns:")
print(schema_content["column_name"].tolist())


fact_content_daily_performance columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']

dim_content columns:
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized

## 3. Verify it with queries (grain, counts, missing values, windows) — plus 5 features and the trap

Three real queries against the **mid-panel month `2026-03`**, then a 5-feature frame, then the
deliberate leakage trap.

In [3]:
MONTH_START = "2026-03-01"
MONTH_END   = "2026-03-31"

# --- Query 1: GRAIN -- prove one row really is one content-day ---
grain_check = con.sql(f"""
    SELECT content_hash_id, client_hash_id, report_date, COUNT(*) AS n
    FROM read_parquet('{DAILY}')
    WHERE report_date BETWEEN '{MONTH_START}' AND '{MONTH_END}'
    GROUP BY content_hash_id, client_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Rows where (content, client, date) repeats -- should be EMPTY if the grain holds:")
print(grain_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows where (content, client, date) repeats -- should be EMPTY if the grain holds:
Empty DataFrame
Columns: [content_hash_id, client_hash_id, report_date, n]
Index: []


**Query 1 result:** an empty table above confirms the grain — one row per content item per day,
no duplicates. If this ever comes back non-empty, the grain claim in Section 1 is wrong and
everything downstream (especially the feature windows) needs re-checking before I go further.

In [4]:
# --- Query 2: ROW COUNT + DATE SPAN for the slice I'm actually developing on ---
count_span = con.sql(f"""
    SELECT
        COUNT(*)                        AS n_rows,
        COUNT(DISTINCT content_hash_id) AS n_content,
        COUNT(DISTINCT client_hash_id)  AS n_clients,
        MIN(report_date)                AS min_date,
        MAX(report_date)                AS max_date
    FROM read_parquet('{DAILY}')
    WHERE report_date BETWEEN '{MONTH_START}' AND '{MONTH_END}'
""").df()
print(count_span)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    n_rows  n_content  n_clients   min_date   max_date
0  9841378     331437         55 2026-03-01 2026-03-31


**Query 2 result:** row count, distinct content items, distinct clients, and the date span for
March 2026, straight from the source table -- not assumed from the lane guide's whole-panel
numbers, which describe all ~17 months, not this one month.

In [5]:
# --- Query 3: AVAILABILITY -- filter with IS TRUE, show how many rows survive ---
avail = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM read_parquet('{DAILY}')
    WHERE report_date BETWEEN '{MONTH_START}' AND '{MONTH_END}'
""").df()
avail["pct_available"] = (avail["ga4_available_rows"] / avail["total_rows"] * 100).round(1)
print(avail)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  ga4_available_rows  pct_available
0     9841378              413966            4.2


**Query 3 result:** `ga4_data_available` filtered with `IS TRUE` (not `= 1` or a plain truthy
check, because the flag can be `NULL` for some early rows, and `NULL` is neither true nor false --
`IS TRUE` is the only comparison that treats `NULL` correctly, as "not available," rather than
silently dropping or keeping it by accident). The surviving percentage tells me how much of March
2026 actually has GA4 engagement data behind it, versus GSC-only rows from clients whose GA4
tracking hadn't started yet.

### 5 features (max), each with "knowable at the decision moment because…"

I'm picking a `decision_date` inside March 2026 for this demo (e.g. `2026-03-15`), and every
feature below is built using **only rows strictly before that date** — that discipline is the
whole point of this section.

In [6]:
DECISION_DATE = "2026-03-15"
PRIOR_START   = "2026-02-15"   # 28 days of prior history before the decision date

features = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,

        -- 1. Prior-window search visibility (28 days strictly before decision_date)
        SUM(gsc_impressions) AS prior_28d_impressions,

        -- 2. Prior-window clicks
        SUM(gsc_clicks) AS prior_28d_clicks,

        -- 3. Prior-window average search position (lower = better)
        AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS prior_28d_avg_position,

        -- 4. Prior-window GA4 sessions (only where GA4 tracking was actually live)
        SUM(ga4_sessions) FILTER (WHERE ga4_data_available IS TRUE) AS prior_28d_sessions,

        -- 5. Coverage flag: was GA4 tracking even live during this prior window?
        BOOL_OR(ga4_data_available) AS ga4_available_in_prior_window

    FROM read_parquet('{DAILY}')
    WHERE report_date >= '{PRIOR_START}' AND report_date < '{DECISION_DATE}'
    GROUP BY content_hash_id, client_hash_id
""").df()

print("Feature frame shape:", features.shape, "-> one row per content item, features from BEFORE decision_date only")
features.head(5)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (337557, 7) -> one row per content item, features from BEFORE decision_date only


,content_hash_id,client_hash_id,prior_28d_impressions,prior_28d_clicks,prior_28d_avg_position,prior_28d_sessions,ga4_available_in_prior_window
0,content_deca2d2dc067396d,client_73cda7b4e4f265ea,1713.0,4.0,4.996075,NaN,<NA>
1,content_31ee224cc7127474,client_73cda7b4e4f265ea,3721.0,28.0,2.850979,NaN,<NA>
2,content_cff6e0d5a8234bc0,client_73cda7b4e4f265ea,4857.0,26.0,5.332348,NaN,<NA>
3,content_e0c5a320b5bbc506,client_73cda7b4e4f265ea,1992.0,0.0,5.215110,NaN,<NA>
4,content_21a85fe2df83d0a2,client_73cda7b4e4f265ea,1332.0,6.0,5.207962,NaN,<NA>


**"Knowable at the decision moment because…" — one line per feature:**

1. `prior_28d_impressions` — search impressions are logged by Google as they happen; every day
   summed here is strictly before `decision_date`, so this total was fully known by then.
2. `prior_28d_clicks` — same reasoning: clicks are observed, past-dated events, none from the
   decision date itself or later.
3. `prior_28d_avg_position` — position is measured per day as search results are served; averaging
   only prior days means no future ranking movement leaks in (rows with no data, `avg_position=0`
   equivalent, are excluded by the `> 0` filter so they don't quietly drag the average down).
4. `prior_28d_sessions` — GA4 sessions are logged as visits happen; restricting to
   `ga4_data_available IS TRUE` avoids treating a client's pre-tracking silence as "zero traffic."
5. `ga4_available_in_prior_window` — this is a coverage fact about the past window itself (was
   tracking live at all), not a measurement that could be influenced by anything after
   `decision_date` — safe by construction.

### The trap: add one label-derived column on purpose, watch the score jump, then remove it

The **label** I'll approximate for this demo: did the page's impressions **fall** from the prior
28-day window to the **next** 28-day window after `decision_date`? That's an actual forward-window
proxy (better than the starter CSV's `is_declining_label`, which only compared two windows that
already existed side by side in the same row).

The **trap**: I'll deliberately add `next_28d_impressions` — a value from the *future* window —
as if it were a feature, watch a quick score look almost perfect, then delete it and report the
honest number instead.

In [7]:
NEXT_END = "2026-04-12"  # 28 days strictly after decision_date

label_frame = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS next_28d_impressions
    FROM read_parquet('{DAILY}')
    WHERE report_date >= '{DECISION_DATE}' AND report_date < '{NEXT_END}'
    GROUP BY content_hash_id, client_hash_id
""").df()

demo = features.merge(label_frame, on=["content_hash_id", "client_hash_id"], how="inner")
demo["declined_next_28d"] = (demo["next_28d_impressions"] < demo["prior_28d_impressions"]).astype(int)

print("Demo frame shape:", demo.shape)
print("Label balance:")
print(demo["declined_next_28d"].value_counts(normalize=True).round(3))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Demo frame shape: (319584, 9)
Label balance:
declined_next_28d
0    0.748
1    0.252
Name: proportion, dtype: float64


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import numpy as np

honest_cols = ["prior_28d_impressions", "prior_28d_clicks", "prior_28d_avg_position", "prior_28d_sessions"]

X_honest = demo[honest_cols].fillna(0)
y = demo["declined_next_28d"]

honest_score = cross_val_score(
    LogisticRegression(max_iter=1000), X_honest, y, cv=5, scoring="roc_auc"
).mean()
print(f"HONEST score (prior-window features only), ROC-AUC: {honest_score:.3f}")

# --- THE TRAP: sneak in a column built from the label's own future window ---
demo["leaky_next_28d_impressions"] = demo["next_28d_impressions"]  # <- this IS the label's raw material

leaky_cols = honest_cols + ["leaky_next_28d_impressions"]
X_leaky = demo[leaky_cols].fillna(0)

leaky_score = cross_val_score(
    LogisticRegression(max_iter=1000), X_leaky, y, cv=5, scoring="roc_auc"
).mean()
print(f"LEAKY score (same features + future-window column), ROC-AUC: {leaky_score:.3f}")
print()
print(f"Jump from adding one future-window column: +{leaky_score - honest_score:.3f}")


HONEST score (prior-window features only), ROC-AUC: 0.819
LEAKY score (same features + future-window column), ROC-AUC: 1.000

Jump from adding one future-window column: +0.181


**What happened, and why:** `leaky_next_28d_impressions` is built from the *same* future window
that defines `declined_next_28d` — giving the model that column is close to handing it the answer
directly, so the score jumps toward a suspiciously perfect number. That jump is the tell. A real
model should never look this good this easily; when it does, the first move is to ask "did I
accidentally include something from after the decision date?", not to celebrate.

**Fix: delete the leaky column and keep only the honest score.**

In [9]:
# Delete the leaky column -- it must never reach a real model.
demo = demo.drop(columns=["leaky_next_28d_impressions"])
assert "leaky_next_28d_impressions" not in demo.columns

print(f"Kept: honest ROC-AUC = {honest_score:.3f}, using only prior-window features.")
print("This is the number that goes in any write-up -- not the leaky one.")


Kept: honest ROC-AUC = 0.819, using only prior-window features.
This is the number that goes in any write-up -- not the leaky one.


## 4. Data limits

**One named limitation of this slice:** March 2026 is a **mid-panel month for the whole release,
but not for every client** — the panel is an unbalanced one (per the `flyrank-data` skill, history
depth varies a lot by client, and only 9 of 70 clients have 12+ months of history). Some clients'
`gsc_data_start`/`ga4_data_start` fall after March 2026 entirely, meaning they contribute zero rows
to this slice, not "quiet" rows. So March 2026 numbers in this notebook describe *whichever
clients had live tracking that month* — not FlyRank's whole client base, and not any single
client's full history. A capstone-grade version of this lane needs to check `dim_clients` coverage
per client before trusting a cross-client comparison, and probably needs per-client windows rather
than one global calendar month for clients with shorter histories.

In [10]:
coverage = con.sql(f"""
    SELECT
        COUNT(*) AS n_clients_total,
        COUNT(*) FILTER (WHERE gsc_data_start <= DATE '{MONTH_START}') AS n_clients_with_march_gsc,
        COUNT(*) FILTER (WHERE ga4_data_start <= DATE '{MONTH_START}') AS n_clients_with_march_ga4
    FROM read_parquet('{DIM_CLIENTS}')
""").df()
print(coverage)
print()
print("-> clients where tracking started AFTER March 2026 contribute nothing to this slice;")
print("   any 'all clients' claim from this notebook would be wrong.")


   n_clients_total  n_clients_with_march_gsc  n_clients_with_march_ga4
0              104                        52                        26

-> clients where tracking started AFTER March 2026 contribute nothing to this slice;
   any 'all clients' claim from this notebook would be wrong.
